In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/annotated/checkpoints/pre_cell_13.pickle

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792aeffb50>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f7930bd5810>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f7930bd56f0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f7a86720f40>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f7a86723550>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792af345b0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792af34190>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792af341f0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792af34a90>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792af34220>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792aeffeb0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f792ae

me:  3
trying: ['features']
me:  13
trying: ['literal_eval']
me:  13
trying: ['np']
me:  0
trying: ['feature']
me:  13
trying: ['pop']


me:  10
trying: ['q_movies']
me:  9
trying: ['df2']


me:  13
trying: ['factor']
me:  2
trying: ['plt']
me:  10


Declaring variable time
Declaring variable pd
Declaring variable np
Declaring variable factor
Declaring variable df1
Declaring variable C
Declaring variable m
Declaring variable weighted_rating
Declaring variable q_movies
Declaring variable pop
Declaring variable plt
Declaring variable features
Declaring variable literal_eval
Declaring variable feature
Declaring variable df2


In [4]:
%%RecordEvent
%%time
### cell 13 ###

import numpy as np

# 1) UDF for director (nested dicts -> CPU)
def get_director(crew):
    for member in crew:
        if member.get('job') == 'Director':
            return member.get('name')
    return np.nan

# Apply on CPU
df2['director'] = df2['crew'].apply(get_director)

# 2) Extract names via a small Python UDF, then GPU‐slice the resulting list column
list_features = ['cast', 'keywords', 'genres']

def get_names(x):
    if isinstance(x, list):
        # pull out just the 'name' field
        return [item.get('name') for item in x]
    return []

for feature in list_features:
    # extract names (CPU-bound)
    df2[feature] = df2[feature].apply(get_names)
    # slice on GPU to keep only the first 3 entries
    df2[feature] = df2[feature].list.slice(0, 3)


AttributeError: 'ListMethods' object has no attribute 'slice'

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/rewritten/o4_mini_high/checkpoints/post_cell_13_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/opt_cell_exec_info_13_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[13], f)


In [ ]:
opt_output = Out.get(4)